# Notebook 01 — Data Loading & Merging
Load all raw sources, fix column names, build FIPS keys, and save two outputs:
- `merged_project_level.csv` — one row per PA project
- `merged_disaster_level.csv` — one row per disaster (aggregated)


In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from utils import data_summary

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'


## 1.1 Load FEMA PA Funded Projects
Key columns: `disasterNumber`, `incidentType`, `stateNumberCode`, `countyCode`,
`damageCategoryCode`, `federalShareObligated`, `projectAmount`, `projectSize`


In [2]:
pa = pd.read_csv(RAW + 'pa_funded_projects.csv', low_memory=False)
data_summary(pa, 'PA Funded Projects')
pa.head(3)



  PA Funded Projects  |  810,656 rows  x  25 cols
Columns with nulls:
  applicantId                                5  (0.0%)
  county                                13,620  (1.7%)



,disasterNumber,declarationDate,incidentType,pwNumber,applicationTitle,applicantId,damageCategoryCode,damageCategoryDescrip,projectStatus,projectProcessStep,...,projectAmount,federalShareObligated,totalObligated,lastObligationDate,firstObligationDate,mitigationAmount,gmProjectId,gmApplicantId,lastRefresh,hash
0,1239,1998-08-26T00:00:00.000Z,Severe Storm(s),1,(PW# 1) IMMEDIATE NEEDS FUNDING,465-19792-00,B,Emergency Protective Measures,Active,Project Closed Out,...,100000.0,75000.00,80340.00,1998-09-15T14:25:07.000Z,1998-09-15T14:25:07.000Z,0.0,1021769,268458,2025-11-27T15:05:59.253Z,addcfded82ae348f46ff034a4564f983a9dea897
1,1239,1998-08-26T00:00:00.000Z,Severe Storm(s),5,(PW# 5) Not Provided,465-19792-00,G,"Parks, Recreational Facilities, and Other Items",Active,Project Closed Out,...,19685.5,14764.13,15461.00,1998-09-23T08:58:52.000Z,1998-09-23T08:58:52.000Z,0.0,1062596,268458,2025-11-27T15:05:59.253Z,05c6c522b930a9c38b52a0c4b0de853e98b4cb75
2,1239,1998-08-26T00:00:00.000Z,Severe Storm(s),7,(PW# 7) Not Provided,465-19792-00,G,"Parks, Recreational Facilities, and Other Items",Active,Project Closed Out,...,26111.0,19583.25,20507.58,1998-09-23T08:58:52.000Z,1998-09-23T08:58:52.000Z,0.0,1062598,268458,2025-11-27T15:05:59.253Z,0addbfed02721821348612482bfe36d8fe587d0f


## 1.2 Load FEMA Disaster Declarations
We only need `incidentBeginDate` and `incidentEndDate` from here — everything else is in PA.
Deduplicate to one row per disaster to avoid multiplying PA rows.


In [3]:
decl = pd.read_csv(RAW + 'disaster_declarations.csv', low_memory=False)
data_summary(decl, 'Disaster Declarations')

# Keep only PA-declared disasters, deduplicate to one row per disasterNumber
decl_pa = decl[decl['paProgramDeclared'] == 1].copy()
decl_unique = (
    decl_pa
    .sort_values('incidentBeginDate')
    .drop_duplicates(subset='disasterNumber', keep='first')
    [['disasterNumber', 'incidentBeginDate', 'incidentEndDate']]
)
print(f'Unique PA disasters: {len(decl_unique):,}')
decl_unique.head(3)



  Disaster Declarations  |  69,766 rows  x  28 cols
Columns with nulls:
  incidentEndDate                          534  (0.8%)
  disasterCloseoutDate                  16,505  (23.7%)
  lastIAFilingDate                      50,333  (72.1%)
  designatedIncidentTypes               47,812  (68.5%)

Unique PA disasters: 4,951


,disasterNumber,incidentBeginDate,incidentEndDate
69765,1,1953-05-02T00:00:00.000Z,1953-05-02T00:00:00.000Z
69764,2,1953-05-15T00:00:00.000Z,1953-05-15T00:00:00.000Z
51472,3,1953-05-29T00:00:00.000Z,1953-05-29T00:00:00.000Z


## 1.3 Merge PA + Declarations


In [4]:
merged = pa.merge(decl_unique, on='disasterNumber', how='left')
print(f'After merge: {merged.shape}')

# Build 5-digit FIPS from stateNumberCode (2-digit) + countyCode (3-digit)
merged['fips'] = (
    merged['stateNumberCode'].astype(str).str.zfill(2) +
    merged['countyCode'].astype(str).str.zfill(3)
)
print('Sample FIPS values:', merged['fips'].head().tolist())


After merge: (810656, 27)
Sample FIPS values: ['48465', '48465', '48465', '22075', '48465']


## 1.4 Load Census Income (S1903)
`skiprows=[1]` drops the human-readable label row so column codes become the headers.
`S1903_C03_001E` = Median household income estimate.


In [5]:
income_raw = pd.read_csv(RAW + 'median_household_income.csv', skiprows=[1], low_memory=False)
print(f'Income shape: {income_raw.shape}')
assert income_raw.shape[0] > 100, 'ERROR: looks like national totals — re-download at county level'

income = income_raw[['GEO_ID', 'S1903_C03_001E']].copy()
income.columns = ['geo_id', 'median_income']
income['fips']         = income['geo_id'].astype(str).str[-5:]
income['median_income'] = pd.to_numeric(income['median_income'], errors='coerce')
income = income[['fips', 'median_income']].dropna()
print(f'Income clean: {income.shape}  |  nulls: {income["median_income"].isna().sum()}')
income.head(3)


Income shape: (3220, 243)
Income clean: (3220, 2)  |  nulls: 0


,fips,median_income
0,01001,58731
1,01003,58320
2,01005,32525


## 1.5 Load Census Poverty Rate (S1701)
`S1701_C03_001E` = Percent of population below the poverty line.


In [6]:
pov_raw = pd.read_csv(RAW + 'poverty_rate.csv', skiprows=[1], low_memory=False)
print(f'Poverty shape: {pov_raw.shape}')
assert pov_raw.shape[0] > 100, 'ERROR: looks like national totals — re-download at county level'

pov = pov_raw[['GEO_ID', 'S1701_C03_001E']].copy()
pov.columns = ['geo_id', 'poverty_rate']
pov['fips']         = pov['geo_id'].astype(str).str[-5:]
pov['poverty_rate'] = pd.to_numeric(pov['poverty_rate'], errors='coerce')
pov = pov[['fips', 'poverty_rate']].dropna()
print(f'Poverty clean: {pov.shape}')
pov.head(3)


Poverty shape: (3220, 369)
Poverty clean: (3220, 2)


,fips,poverty_rate
0,01001,15.2
1,01003,10.4
2,01005,30.7


## 1.6 Load FEMA National Risk Index
Provides population per county + overall hazard risk score.
We use NRI population instead of the transposed Census xlsx file.


In [7]:
nri_raw = pd.read_csv(RAW + 'national_risk_index.csv', low_memory=False)
print(f'NRI shape: {nri_raw.shape}')

nri = nri_raw[['STCOFIPS', 'POPULATION', 'RISK_SCORE', 'RISK_RATNG']].copy()
nri.columns = ['fips', 'population', 'risk_score', 'risk_rating']
nri['fips']       = nri['fips'].astype(str).str.zfill(5)
nri['population'] = pd.to_numeric(nri['population'], errors='coerce')
nri['risk_score'] = pd.to_numeric(nri['risk_score'], errors='coerce')
print(f'NRI clean: {nri.shape}')
nri.head(3)


NRI shape: (3232, 465)
NRI clean: (3232, 4)


,fips,population,risk_score,risk_rating
0,01001,58764,57.569975,Relatively Low
1,01003,231365,96.723919,Relatively High
2,01005,25160,48.123410,Relatively Low


## 1.7 Combine County-Level Features & Merge Into Project Data


In [8]:
# Combine census + NRI on FIPS
county_features = income.merge(pov, on='fips', how='outer').merge(nri, on='fips', how='outer')
print(f'County features: {county_features.shape}')

# Merge into project-level data
merged = merged.merge(county_features, on='fips', how='left')
print(f'Final project-level shape: {merged.shape}')
data_summary(merged, 'Merged Project-Level')


County features: (3241, 6)
Final project-level shape: (810656, 33)

  Merged Project-Level  |  810,656 rows  x  33 cols
Columns with nulls:
  applicantId                                5  (0.0%)
  county                                13,620  (1.7%)
  median_income                         20,890  (2.6%)
  poverty_rate                          20,890  (2.6%)
  population                            26,043  (3.2%)
  risk_score                            49,701  (6.1%)
  risk_rating                           26,043  (3.2%)



## 1.8 Create Disaster-Level Aggregate
One row per disaster — this is the label for the Level 1 (strategic) model.


In [9]:
disaster_level = merged.groupby('disasterNumber').agg(
    total_federal_share = ('federalShareObligated', 'sum'),
    n_projects          = ('federalShareObligated', 'count'),
    n_counties          = ('fips', 'nunique'),
    # Disaster-level attributes (same value for all rows — take first)
    incidentType        = ('incidentType', 'first'),
    stateCode           = ('stateNumberCode', 'first'),
    stateAbbreviation   = ('stateAbbreviation', 'first'),
    declarationDate     = ('declarationDate', 'first'),
    incidentBeginDate   = ('incidentBeginDate', 'first'),
    incidentEndDate     = ('incidentEndDate', 'first'),
    # County-level features: sum population, median everything else
    population          = ('population', 'sum'),
    median_income       = ('median_income', 'median'),
    poverty_rate        = ('poverty_rate', 'median'),
    risk_score          = ('risk_score', 'median'),
).reset_index()

print(f'Disaster-level shape: {disaster_level.shape}')
disaster_level.head(3)


Disaster-level shape: (1766, 14)


,disasterNumber,total_federal_share,n_projects,n_counties,incidentType,stateCode,stateAbbreviation,declarationDate,incidentBeginDate,incidentEndDate,population,median_income,poverty_rate,risk_score
0,1239,7950369.84,276,6,Severe Storm(s),48,TX,1998-08-26T00:00:00.000Z,1998-08-22T00:00:00.000Z,1998-08-31T00:00:00.000Z,48941150.0,46147.0,20.3,76.081425
1,1257,32229051.37,1760,38,Flood,48,TX,1998-10-21T00:00:00.000Z,1998-10-17T00:00:00.000Z,1998-11-15T00:00:00.000Z,788095965.0,57157.0,14.1,86.482188
2,1260,9485830.00,212,35,Severe Storm(s),47,TN,1999-01-15T00:00:00.000Z,1998-12-23T00:00:00.000Z,1998-12-29T00:00:00.000Z,8042506.0,49485.0,15.8,60.909669


## 1.9 Save Processed Files


In [10]:
merged.to_csv(PROCESSED + 'merged_project_level.csv', index=False)
disaster_level.to_csv(PROCESSED + 'merged_disaster_level.csv', index=False)

print('Saved:')
print(f'  merged_project_level.csv   -> {merged.shape}')
print(f'  merged_disaster_level.csv  -> {disaster_level.shape}')


Saved:
  merged_project_level.csv   -> (810656, 33)
  merged_disaster_level.csv  -> (1766, 14)
